# Gold Layer Queries

Análises SQL para o esquema estrela `gold` (`dim_prdt`, `dim_tmp`, `dim_cat`, `ft_vnd`).

- Execute com o PostgreSQL do projeto em execução (Docker ou local).
- Certifique-se de que as variáveis de ambiente (`POSTGRES_*`) estejam alinhadas com suas credenciais.
- Utilize as funções auxiliares definidas abaixo para rodar e exibir os resultados.


In [61]:
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor
from IPython.display import display

# Formatação padrão para floats
pd.options.display.float_format = lambda value: f"{value:,.2f}" if isinstance(value, float) else f"{value}"

# Configuração do banco de dados
# Usando valores padrão do projeto (postgres/postgres)
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "amazon_sales",
    "user": "postgres",
    "password": "postgres",
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    print(
        f"✅ Conectado ao banco '{DB_CONFIG['database']}' em {DB_CONFIG['host']}:{DB_CONFIG['port']} como usuário {DB_CONFIG['user']}"
    )
except psycopg2.Error as exc:
    raise RuntimeError("Não foi possível conectar ao banco PostgreSQL. Verifique as credenciais e se o serviço está ativo.") from exc


✅ Conectado ao banco 'amazon_sales' em localhost:5432 como usuário postgres


In [62]:
def run_query(sql: str) -> pd.DataFrame:
    """Executa uma consulta SQL e retorna os resultados em um DataFrame."""
    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        cursor.execute(sql)
        rows = cursor.fetchall()
    return pd.DataFrame(rows)


def show(df: pd.DataFrame, max_rows: int = 10) -> None:
    """Mostra até `max_rows` linhas e um resumo compacto (padrão = 10)."""
    if df.empty:
        print("Resultado vazio.")
        return
    display(df.head(max_rows))
    total = len(df)
    if total > max_rows:
        print(f"\n→ mostrando {max_rows} de {total:,} linhas × {len(df.columns)} colunas")
    else:
        print(f"\n→ {total:,} linhas × {len(df.columns)} colunas")


## Verificação rápida do esquema `gold`
Confirme que as tabelas possuem dados antes de seguir com as análises.


In [63]:
sanity_sql = """
SELECT 'dim_prdt' AS tabela, COUNT(*) AS linhas FROM gold.dim_prdt
UNION ALL
SELECT 'dim_tmp'  AS tabela, COUNT(*) AS linhas FROM gold.dim_tmp
UNION ALL
SELECT 'dim_cat' AS tabela, COUNT(*) AS linhas FROM gold.dim_cat
UNION ALL
SELECT 'ft_vnd'   AS tabela, COUNT(*) AS linhas FROM gold.ft_vnd
ORDER BY tabela;
"""

sanity_df = run_query(sanity_sql)
show(sanity_df, max_rows=10)


,tabela,linhas
0,dim_cat,22
1,dim_prdt,15938
2,dim_tmp,6
3,ft_vnd,11559



→ 4 linhas × 2 colunas


## 1. Desempenho por categoria

**Objetivo:** medir o tamanho, a qualidade percebida e o potencial promocional de cada categoria.

**Por que executar:** revela rapidamente quais categorias sustentam a maior parte da receita e onde campanhas de promoção ou expansão de portfólio podem gerar impacto.


In [64]:
categoria_sql = """
SELECT
    p.categoria,
    SUM(f.unidades_vendidas) AS total_unidades,
    SUM(f.receita_estimada) AS receita_total,
    AVG(f.rating) AS rating_medio,
    AVG(f.quality_score) AS quality_score_medio,
    SUM(CASE WHEN p.is_promotable THEN 1 ELSE 0 END) AS produtos_promoviveis,
    100.0 * SUM(CASE WHEN p.is_promotable THEN 1 ELSE 0 END) / NULLIF(COUNT(DISTINCT p.prdt_key), 0) AS pct_promovivel,
    SUM(CASE WHEN p.best_seller_badge THEN f.receita_estimada ELSE 0 END) AS receita_best_seller
FROM gold.ft_vnd f
JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
GROUP BY p.categoria
ORDER BY receita_total DESC;
"""

categoria_df = run_query(categoria_sql)
show(categoria_df)


,categoria,total_unidades,receita_total,rating_medio,quality_score_medio,produtos_promoviveis,pct_promovivel,receita_best_seller
0,Power,26849150,532227302.50,4.7583896103896104,87.5307636363636364,0,0E-20,4603200.00
1,Audio,2011600,213649711.75,4.3172143207454635,80.2192888670917116,0,0E-20,50144310.25
2,Printing,913200,101660671.50,4.5465661641541039,77.0859798994974874,0,0E-20,8183320.00
3,Mobile,1580000,92337691.75,4.3686256781193490,83.3557685352622061,0,0E-20,11613765.50
4,Accessory,1109250,76191255.00,4.4518110236220472,80.4784881889763780,0,0E-20,7663993.00
5,Camera,659650,71579634.00,4.5185142857142857,82.9943085714285714,0,0E-20,8013458.00
6,Laptop,930200,62374055.00,4.4270753512132822,77.8707535121328225,0,0E-20,6203460.00
7,Storage,509000,52557450.50,4.5109958506224066,84.1398755186721992,0,0E-20,4190430.00
8,Other,915200,51198149.00,4.5229744728079911,81.7892452830188679,0,0E-20,6428062.50
9,Printing Supplies,548800,36312147.25,4.5968354430379747,84.2272468354430380,0,0E-20,5021700.00



→ mostrando 10 de 22 linhas × 8 colunas


## 2. Top 3 faixas de preço dentro de cada categoria

**Objetivo:** descobrir em quais tickets cada categoria mais fatura e qual peso (%) esses tickets têm no resultado da categoria.

**Por que executar:** ajuda a planejar sortimento e promoções focando nas faixas que realmente movem receita em cada categoria, sem olhar apenas para o total agregado.


In [ ]:
categoria_preco_sql = """
WITH base AS (
    SELECT
        p.categoria,
        p.faixa_preco,
        SUM(f.receita_estimada) AS receita_total,
        SUM(f.unidades_vendidas) AS unidades_total
    FROM gold.ft_vnd f
    JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
    GROUP BY p.categoria, p.faixa_preco
),
ranked AS (
    SELECT
        b.*,
        ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY receita_total DESC) AS pos_receita,
        SUM(receita_total) OVER (PARTITION BY categoria) AS receita_categoria
    FROM base b
)
SELECT
    categoria,
    faixa_preco,
    receita_total,
    ROUND(100.0 * receita_total / NULLIF(receita_categoria, 0), 2) AS pct_receita_categoria,
    unidades_total,
    ROUND(receita_total / NULLIF(unidades_total, 0), 2) AS ticket_medio
FROM ranked
WHERE pos_receita <= 3
ORDER BY receita_categoria DESC, categoria, pos_receita;
"""

categoria_preco_df = run_query(categoria_preco_sql)
show(categoria_preco_df)


,categoria,faixa_preco,receita_total,unidades_total,preco_medio,ticket_medio
0,Accessory,Premium ($100-200),33218072.50,269000,143.0752367688022284,123.4872583643122677
1,Accessory,High-End ($200-500),14091951.00,49700,306.6815625000000000,283.5402615694164990
2,Accessory,Economy ($20-50),12023454.00,358100,35.4069285714285714,33.5756883552080424
3,Accessory,Mid-Range ($50-100),8635970.50,115200,78.4118604651162791,74.9650217013888889
4,Accessory,Luxury ($500+),4278022.50,5550,755.0386666666666667,770.8148648648648649
5,Accessory,Budget (< $20),3943784.50,311700,13.0682162162162162,12.6525008020532563
6,Audio,Premium ($100-200),127356647.00,937750,132.9211847389558233,135.8108739002932551
7,Audio,High-End ($200-500),32379247.75,108450,307.0753333333333333,298.5638335638543107
8,Audio,Mid-Range ($50-100),25913350.75,331850,81.1462559241706161,78.0875418110592135
9,Audio,Economy ($20-50),17633923.25,463500,35.4899350649350649,38.0451418554476807



→ mostrando 10 de 116 linhas × 6 colunas


## 3. Faixas de preço mais fortes (geral)

**Objetivo:** avaliar quais faixas de preço concentram produtos, receita e satisfação no portfólio inteiro.

**Por que executar:** dá uma visão direta de quais tickets sustentam o faturamento total, útil para definir estratégia de entrada/saída de produtos ou promoções.


In [ ]:
faixa_preco_geral_sql = """
SELECT
    p.faixa_preco,
    COUNT(DISTINCT p.prdt_key) AS produtos,
    SUM(f.unidades_vendidas) AS unidades_total,
    SUM(f.receita_estimada) AS receita_total,
    ROUND(SUM(f.receita_estimada) / NULLIF(SUM(f.unidades_vendidas), 0), 2) AS ticket_medio,
    AVG(f.rating) AS rating_medio,
    AVG(f.quality_score) AS quality_score_medio
FROM gold.ft_vnd f
JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
GROUP BY p.faixa_preco
ORDER BY receita_total DESC;
"""

faixa_preco_geral_df = run_query(faixa_preco_geral_sql)
show(faixa_preco_geral_df)


,faixa_desconto,linhas_fato,unidades_total,receita_total,desconto_medio,rating_medio
0,Sem desconto,11559,38283200,1411036999.25,0E-24,4.4906894162480513



→ 1 linhas × 6 colunas


## 4. "Hidden gems": alta qualidade, baixa receita

**Objetivo:** encontrar os 10 produtos com melhor qualidade percebida dentro das faixas de menor receita.

**Por que executar:** ajuda a direcionar ações simples de divulgação e promoções específicas para itens promissores que ainda vendem pouco.


In [67]:
hidden_gems_sql = """
WITH limites AS (
    SELECT
        percentile_cont(0.75) WITHIN GROUP (ORDER BY f.quality_score) AS q75_quality,
        percentile_cont(0.25) WITHIN GROUP (ORDER BY f.receita_estimada) AS q25_receita
    FROM gold.ft_vnd f
    WHERE f.quality_score IS NOT NULL
      AND f.receita_estimada IS NOT NULL
)
SELECT
    p.asin,
    p.titulo,
    p.marca,
    p.categoria,
    f.quality_score,
    f.receita_estimada,
    f.unidades_vendidas,
    f.rating
FROM gold.ft_vnd f
JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
CROSS JOIN limites l
WHERE f.quality_score >= l.q75_quality
  AND f.receita_estimada <= l.q25_receita
ORDER BY f.quality_score DESC, f.receita_estimada ASC
LIMIT 10;
"""

hidden_gems_df = run_query(hidden_gems_sql)
show(hidden_gems_df)


,asin,titulo,marca,categoria,quality_score,receita_estimada,unidades_vendidas,rating
0,B08W1WF6ZS,"UGREEN Cat 8 Ethernet Cable 3FT, Flat High Spe...",ugreen,Networking,98.00,2995.00,500,4.80
1,B00L1LM5UU,"D'Addario Electric Guitar Strings, NYXL Nickel...",addario,Other,98.00,3897.00,300,4.80
2,B08Q87MF19,Duracell Coppertop AA Batteries with Power Boo...,duracell,Power,98.00,4450.00,200,4.80
3,B07H9J1YXN,[Older Version] SanDisk 64GB Extreme PRO SDXC ...,unknown,Storage,98.00,4713.00,300,4.80
4,B001TDKOOY,Pentel EnerGel Deluxe RTX Retractable Gel Pens,pentel,Office Supplies,98.00,4790.00,1000,4.80
5,B0090YJBYS,HP Printer Paper | 8.5 x 11 Paper | Office 20 ...,hp,Printing,98.00,4794.00,600,4.80
6,B00ECEVGN0,SanDisk 128GB Extreme PRO CompactFlash Memory ...,sandisk,Storage,98.00,4959.00,200,4.80
7,B01J5RHBQ4,SanDisk Extreme Pro 32GB SDHC UHS-I Card (SDSD...,sandisk,Other,98.00,5319.00,300,4.80
8,B089C73T72,"SAMSUNG 870 QVO SATA III SSD 1TB 2.5"" Internal...",samsung,Laptop,98.00,5500.00,50,4.80
9,B07NJ8MS3K,GE 3-Outlet Power Strip Surge Protector 15 Ft ...,ge,Power,98.00,6115.00,500,4.80



→ 10 linhas × 8 colunas


## 5. Pipeline de produtos promovíveis por marca

**Objetivo:** ranquear marcas com volume consistente de itens elegíveis para campanhas e promoções.

**Por que executar:** suporta negociações comerciais e planejamento de mídia, destacando as 10 marcas que concentram o maior potencial de receita promocionável.


In [68]:
promoviveis_sql = """
SELECT
    p.marca,
    COUNT(DISTINCT p.prdt_key) AS total_produtos,
    SUM(CASE WHEN p.is_promotable THEN 1 ELSE 0 END) AS produtos_promoviveis,
    SUM(CASE WHEN p.is_promotable THEN COALESCE(f.receita_estimada, 0) ELSE 0 END) AS receita_promovivel,
    AVG(f.rating) AS rating_medio,
    SUM(COALESCE(f.unidades_vendidas, 0)) AS unidades_total
FROM gold.dim_prdt p
LEFT JOIN gold.ft_vnd f ON p.prdt_key = f.prdt_key
GROUP BY p.marca
HAVING COUNT(DISTINCT p.prdt_key) >= 5
ORDER BY receita_promovivel DESC
LIMIT 10;
"""

promoviveis_df = run_query(promoviveis_sql)
show(promoviveis_df)


,marca,total_produtos,produtos_promoviveis,receita_promovivel,rating_medio,unidades_total
0,acdelco,8,0,0,4.6750000000000000,24800
1,acer,288,0,0,4.4117647058823529,128000
2,addario,64,0,0,4.8609375000000000,8700
3,adidas,9,0,0,4.7333333333333333,10000
4,ainope,17,0,0,4.5941176470588235,71000
5,alienware,7,0,0,4.2600000000000000,800
6,amazfit,11,0,0,4.2727272727272727,9300
7,amazon,205,0,0,4.5512315270935961,1514300
8,amd,30,0,0,4.7321428571428571,30500
9,8bitdo,23,0,0,4.4578947368421053,9900



→ 10 linhas × 6 colunas


## 6. Datas campeãs de receita por ano

**Objetivo:** destacar os dias de maior faturamento em cada ano, acompanhando volume e satisfação.

**Por que executar:** revela quais datas puxam o resultado anual (ex.: sazonalidade ou eventos) e facilita preparar estoque, mídia e equipe para repetir o bom desempenho.


In [69]:
serie_temporal_sql = """
WITH diario AS (
    SELECT
        t.data,
        EXTRACT(YEAR FROM t.data) AS ano,
        t.nome_dia_semana,
        SUM(f.receita_estimada) AS receita_total,
        SUM(f.unidades_vendidas) AS unidades_total,
        AVG(f.rating) AS rating_medio
    FROM gold.ft_vnd f
    JOIN gold.dim_tmp t ON f.tmp_key = t.tmp_key
    GROUP BY t.data, t.nome_dia_semana
),
ranked AS (
    SELECT
        d.*,
        ROW_NUMBER() OVER (PARTITION BY ano ORDER BY receita_total DESC) AS pos_receita,
        SUM(receita_total) OVER (PARTITION BY ano) AS receita_ano
    FROM diario d
)
SELECT
    ano,
    data,
    nome_dia_semana,
    receita_total,
    ROUND(100.0 * receita_total / NULLIF(receita_ano, 0), 2) AS pct_receita_ano,
    unidades_total,
    rating_medio
FROM ranked
WHERE pos_receita <= 10
ORDER BY ano, pos_receita;
"""

serie_temporal_df = run_query(serie_temporal_sql)
show(serie_temporal_df)


,ano,data,nome_dia_semana,receita_total,pct_receita_ano,unidades_total,rating_medio
0,2025,2025-08-21,Thursday,876514980.25,62.12,15834950,4.4875183123351890
1,2025,2025-08-27,Wednesday,206646146.25,14.64,10670250,4.5111441307578009
2,2025,2025-08-29,Friday,143240607.75,10.15,5310500,4.5863354037267081
3,2025,2025-08-24,Sunday,71268109.25,5.05,2402150,4.4424169475204622
4,2025,2025-08-30,Saturday,66488762.00,4.71,2358850,4.5707589285714286
5,2025,2025-08-25,Monday,46878393.75,3.32,1706500,4.4544964028776978



→ 6 linhas × 7 colunas


## 7. Origem do preço usada nas vendas

**Objetivo:** comparar quanto cada tipo de preço (capturado do site ou calculado pelo ETL) contribui para vendas e receita.

**Por que executar:** confirma se dependemos demais de preços imputados e se esses registros têm qualidade semelhante aos preços originais. Caso um tipo de preço tenha nota baixa ou pouca receita, é sinal de ajuste no cálculo.


In [ ]:
dependencia_preco_sql = """
WITH base AS (
    SELECT
        p.categoria,
        f.origem_preco,
        SUM(f.receita_estimada) AS receita_total,
        SUM(f.unidades_vendidas) AS unidades_total
    FROM gold.ft_vnd f
    JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
    GROUP BY p.categoria, f.origem_preco
),
pivot AS (
    SELECT
        categoria,
        SUM(CASE WHEN origem_preco = 'original' THEN receita_total ELSE 0 END) AS receita_original,
        SUM(CASE WHEN origem_preco <> 'original' THEN receita_total ELSE 0 END) AS receita_imputada,
        SUM(CASE WHEN origem_preco = 'original' THEN unidades_total ELSE 0 END) AS unidades_originais,
        SUM(CASE WHEN origem_preco <> 'original' THEN unidades_total ELSE 0 END) AS unidades_imputadas
    FROM base
    GROUP BY categoria
),
resumo AS (
    SELECT
        categoria,
        receita_original + receita_imputada AS receita_total,
        receita_original,
        receita_imputada,
        unidades_originais,
        unidades_imputadas,
        ROUND(100.0 * receita_imputada / NULLIF(receita_original + receita_imputada, 0), 2) AS pct_receita_imputada
    FROM pivot
)
SELECT
    categoria,
    receita_total,
    receita_original,
    receita_imputada,
    pct_receita_imputada,
    unidades_originais,
    unidades_imputadas
FROM resumo
ORDER BY receita_total DESC
LIMIT 15;
"""

dependencia_preco_df = run_query(dependencia_preco_sql)
show(dependencia_preco_df)


,origem_preco,registros,pct_registros,unidades_total,pct_unidades,receita_total,pct_receita,preco_medio,rating_medio,quality_score_medio
0,original,9340,80.80,35371500,92.39,1096823590.00,77.73,121.6686370449678801,4.4968167202572347,82.4358092175777063
1,imputed_brand_cat,1781,15.41,2506000,6.55,282804750.25,20.04,220.1753902302077485,4.4463743676222597,80.4547498594716133
2,imputed_brand,156,1.35,133450,0.35,22179055.00,1.57,221.9240384615384615,4.4762820512820513,79.3103846153846154
3,imputed_category,282,2.44,272250,0.71,9229604.00,0.65,31.7281560283687943,4.5758007117437722,84.2806049822064057



→ 4 linhas × 10 colunas


## 8. Impacto de disponibilidade e badges

**Objetivo:** comparar combinações de disponibilidade, badge de best seller e patrocínio e medir quanto cada cenário traz de faturamento.

**Por que executar:** mostra quais sinais visíveis aos clientes (estoque, badge, anúncio patrocinado) realmente puxam receita, ajudando a decidir onde investir reposição e mídia.


In [ ]:
badges_sql = """
WITH base AS (
    SELECT
        CASE WHEN p.disponivel_compra THEN 'Disponível' ELSE 'Indisponível' END AS disponibilidade,
        CASE WHEN p.best_seller_badge THEN 'Com best seller' ELSE 'Sem best seller' END AS best_seller,
        CASE WHEN p.sponsored_badge THEN 'Patrocinado' ELSE 'Não patrocinado' END AS patrocinado,
        SUM(f.receita_estimada) AS receita_total,
        SUM(f.unidades_vendidas) AS unidades_total,
        COUNT(DISTINCT p.prdt_key) AS produtos
    FROM gold.ft_vnd f
    JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
    GROUP BY 1, 2, 3
),
totais AS (
    SELECT SUM(receita_total) AS receita_geral FROM base
)
SELECT
    disponibilidade,
    best_seller,
    patrocinado,
    receita_total,
    ROUND(100.0 * receita_total / NULLIF(t.receita_geral, 0), 2) AS pct_receita,
    unidades_total,
    produtos
FROM base
CROSS JOIN totais t
ORDER BY receita_total DESC;
"""

badges_df = run_query(badges_sql)
show(badges_df)


,contexto,receita_total,pct_receita,unidades_total,produtos,receita_por_produto
0,Disponível | Sem badge | Patrocinado,629581409.50,44.62,27035700,3466,181644.95
1,Disponível | Sem badge | Não patrocinado,362188764.00,25.67,6058950,5192,69759.01
2,Indisponível | Sem badge | Não patrocinado,276165279.50,19.57,2605700,2569,107499.14
3,Indisponível | Badge best seller | Não patroci...,51864027.75,3.68,574150,54,960444.96
4,Disponível | Badge best seller | Não patrocinado,43896493.00,3.11,1369200,141,311322.65
5,Disponível | Badge best seller | Patrocinado,41380529.50,2.93,536050,55,752373.26
6,Indisponível | Sem badge | Patrocinado,5960496.00,0.42,103450,82,72688.98



→ 7 linhas × 6 colunas


## Encerramento


In [72]:
try:
    conn.close()
    print("Conexão encerrada.")
except NameError:
    print("Conexão não estava definida.")


Conexão encerrada.
